# Git and Version Control — Examples

This notebook demonstrates Git concepts using a **temporary local repository**. It will not push anything to GitHub and will not change your existing repository.

**Prerequisite:** Git must be installed and available on your PATH. If Git is not installed, read `README.md` first.

The examples use Python's `subprocess` module to run Git commands and show their output.

In [ ]:
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

GIT = shutil.which("git")
if GIT is None:
    raise RuntimeError("Git was not found. Install Git and restart Jupyter.")
print(subprocess.run([GIT, "--version"], capture_output=True, text=True, check=True).stdout.strip())

## 1. Configure identity for this temporary repository only
We set local configuration so global settings are not changed.

In [ ]:
workspace = Path(tempfile.mkdtemp(prefix="git-learning-"))
repo = workspace / "demo-repo"
repo.mkdir()

def git(*args, check=True):
    result = subprocess.run(
        [GIT, *args], cwd=repo, capture_output=True, text=True
    )
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr or result.stdout)
    output = (result.stdout + result.stderr).strip()
    print(f"$ git {' '.join(args)}")
    print(output if output else f"(exit code {result.returncode})")
    return result

git("init", "-b", "main")
git("config", "user.name", "Learning User")
git("config", "user.email", "learning@example.com")
git("status", "--short", "--branch")

## 2. Create and commit a README
The staging area lets you choose exactly which changes enter the next commit.

In [ ]:
(repo / "README.md").write_text("# Sample ML Project\n\nA tiny repository for learning Git.\n", encoding="utf-8")
git("status", "--short")
git("add", "README.md")
git("diff", "--staged")
git("commit", "-m", "Add project README")
git("log", "--oneline", "--decorate")

## 3. Compare unstaged and staged changes

In [ ]:
readme_path = repo / "README.md"
readme_path.write_text(readme_path.read_text(encoding="utf-8") + "\n## Goals\n- Track changes\n", encoding="utf-8")
(repo / "notes.txt").write_text("This file is a practice note.\n", encoding="utf-8")

git("status", "--short")
git("diff")                 # unstaged tracked-file changes
git("add", "README.md")
git("diff", "--staged")     # changes selected for next commit
git("status", "--short")

## 4. Commit and inspect history

In [ ]:
git("commit", "-m", "Document project goals")
git("log", "--oneline", "--graph", "--decorate", "--all")
git("show", "--stat", "HEAD")

## 5. Branching and merging

In [ ]:
git("switch", "-c", "feature/add-data-dictionary")
(repo / "data_dictionary.md").write_text(
    "# Data Dictionary\n\n- `sample_id`: unique row identifier\n", encoding="utf-8"
)
git("add", "data_dictionary.md")
git("commit", "-m", "Add data dictionary")
git("switch", "main")
git("merge", "--no-edit", "feature/add-data-dictionary")
git("log", "--oneline", "--graph", "--decorate", "--all")
git("branch")

## 6. `.gitignore` and status

In [ ]:
(repo / ".gitignore").write_text(
    ".venv/\n__pycache__/\n.ipynb_checkpoints/\n.env\n*.joblib\n",
    encoding="utf-8"
)
(repo / ".env").write_text("EXAMPLE_TOKEN=not-a-real-secret\n", encoding="utf-8")
git("add", ".gitignore")
git("status", "--short")
print("\nNotice: .env is untracked and ignored; it should not be committed.")

## 7. Safe undo demonstrations

The next cells demonstrate unstaging and reverting a commit. They do not use `reset --hard` or modify any repository outside this temporary folder.

In [ ]:
(repo / "draft.md").write_text("draft content\n", encoding="utf-8")
git("add", "draft.md")
git("status", "--short")
git("restore", "--staged", "draft.md")
git("status", "--short")
print("The file remains on disk; it was only removed from the staging area.")

In [ ]:
git("add", ".gitignore")
git("commit", "-m", "Ignore local environments and secrets")
git("log", "--oneline", "--graph", "--decorate", "--all")
print(f"Temporary repository: {repo}")
print("When finished, you may delete this temporary directory manually.")

## 8. Practice commands (run manually in a separate practice repository)

```bash
git status
git diff
git diff --staged
git log --oneline --graph --decorate --all
git branch
git switch -c feature/my-change
git merge feature/my-change
git remote -v
git fetch origin
```

Do not run `git push`, `git reset --hard`, or `git clean -fd` unless you understand the target repository and consequences.

## Key takeaways

- Git is local-first; a commit does not automatically upload changes.
- Stage intentionally and inspect the staged diff.
- Branches make parallel work manageable.
- `.gitignore` does not remove files already tracked.
- Protect credentials and sensitive datasets; rotate secrets if exposed.